# QTrans—平衡二分类冻结嵌套正式评估

本 Notebook 只读取两套 `frozen_nested_selection.json`，不运行候选搜索，也不允许修改冻结配置。每个模型使用一组全新训练种子 `142, 152, 162, 172, 182`。

- UCR Wafer：在全部 194 个固定平衡官方 TRAIN 样本上训练冻结轮数，然后评价 1330 个固定平衡官方 TEST 样本；
- SECOM：每个外折使用该折冻结配置，在全部外层开发样本上训练冻结轮数，测试外折后拼接 208 个样本的 OOF 预测；
- 不使用测试分数选择 epoch、检查点或候选配置。

## 发表边界

旧版模型的测试汇总已经被查看，所以本轮是合规的冻结后再评估，但不是完全未见过数据集的事前预注册确证。如果新结果领先，最强的外部确证仍应来自一个尚未查看测试结果的新数据集。

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch

from qcs_balanced_binary import paired_comparisons
from qcs_balanced_nested_formal import (
    NESTED_FORMAL_SEEDS, audit_nested_formal, formal_progress,
    require_formal_complete, run_secom_nested_formal,
    run_ucr_nested_formal,
)

pd.set_option('display.max_columns', 100)
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'qcs_balanced_nested_formal.py').exists():
    raise RuntimeError('请从 /root/xxx/autodl 目录打开本 Notebook')
UCR_DIR = PROJECT_DIR / 'data' / 'raw' / 'Wafer'
SECOM_DIR = PROJECT_DIR / 'data' / 'raw' / 'secom'
TUNING_ROOT = PROJECT_DIR / 'artifacts' / 'balanced_binary_nested_tuning'
UCR_FROZEN = TUNING_ROOT / 'ucr_wafer' / 'frozen_nested_selection.json'
SECOM_FROZEN = TUNING_ROOT / 'secom' / 'frozen_nested_selection.json'
ARTIFACT_ROOT = PROJECT_DIR / 'artifacts' / 'balanced_binary_nested_formal'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for path in (UCR_FROZEN, SECOM_FROZEN):
    if not path.exists():
        raise FileNotFoundError(path)
display(pd.DataFrame([{'device': str(DEVICE), 'formal_seeds': NESTED_FORMAL_SEEDS, 'artifacts': str(ARTIFACT_ROOT)}]))

## 1. 冻结配置、数据和参数量强制审计

审计会核对冻结协议、数据集名、`test_evaluated=False`、候选ID与配置逐字段一致性、固定轮数和1%参数量边界。任一项不符合都会停止。

In [ ]:
ucr_audit = audit_nested_formal('ucr_wafer', UCR_DIR, UCR_FROZEN)
display(pd.DataFrame([ucr_audit['dataset']]))
display(ucr_audit['selections'])
display(ucr_audit['parameters'])
print('UCR frozen SHA-256:', ucr_audit['frozen_sha256'])

secom_audit = audit_nested_formal('secom', SECOM_DIR, SECOM_FROZEN)
display(pd.DataFrame([secom_audit['dataset']]))
display(secom_audit['selections'])
display(secom_audit['parameters'])
print('SECOM frozen SHA-256:', secom_audit['frozen_sha256'])

## 2. UCR Wafer 冻结正式评估

共 `4模型 × 5新种子 = 20` 个任务。每个任务保存最终检查点、训练曲线、测试预测、混淆矩阵和包含冻结文件哈希的完成签名。

In [ ]:
ucr_before = formal_progress(ARTIFACT_ROOT, 'ucr_wafer')
display(ucr_before.groupby('model')['complete'].agg(['sum', 'count']))
print(f"UCR 已完成 {int(ucr_before.complete.sum())}/{len(ucr_before)}")

`MAX_JOBS_UCR=None` 表示完成所有剩余任务。首次可设为 `1` 验证环境，之后恢复 `None`。

In [ ]:
MAX_JOBS_UCR = None
ucr_results = run_ucr_nested_formal(
    data_dir=UCR_DIR, frozen_selection_path=UCR_FROZEN,
    artifact_dir=ARTIFACT_ROOT, seeds=NESTED_FORMAL_SEEDS,
    balance_seed=2026, max_jobs=MAX_JOBS_UCR, device=DEVICE,
)
print(f'UCR 当前结果 {len(ucr_results)}/20')
display(ucr_results)

In [ ]:
ucr_after = formal_progress(ARTIFACT_ROOT, 'ucr_wafer')
require_formal_complete(ucr_after)
UCR_RESULT_DIR = ARTIFACT_ROOT / 'ucr_wafer'
ucr_results = pd.read_csv(UCR_RESULT_DIR / 'results.csv')
ucr_summary = pd.read_csv(UCR_RESULT_DIR / 'summary.csv')
display(ucr_summary)
ucr_paired_acc = paired_comparisons(ucr_results, metric='accuracy')
ucr_paired_f1 = paired_comparisons(ucr_results, metric='macro_f1')
ucr_paired_acc.to_csv(UCR_RESULT_DIR / 'paired_accuracy.csv', index=False)
ucr_paired_f1.to_csv(UCR_RESULT_DIR / 'paired_macro_f1.csv', index=False)
display(ucr_paired_acc)
display(ucr_paired_f1)

In [ ]:
ax = ucr_summary.set_index('model')[['accuracy_mean', 'macro_f1_mean']].plot.bar(
    figsize=(9, 4), ylim=(0, 1), rot=15, grid=True
)
ax.set_ylabel('score')
ax.set_title('Frozen nested UCR Wafer formal evaluation')
plt.tight_layout()
plt.savefig(UCR_RESULT_DIR / 'formal_accuracy_macro_f1.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. SECOM 冻结外折 OOF 正式评估

共 `5外折 × 4模型 × 5新种子 = 100` 个训练任务。论文统计单位是每个种子覆盖208个样本一次的 OOF 结果，不把5个外折伪装成独立重复实验。

In [ ]:
secom_before = formal_progress(ARTIFACT_ROOT, 'secom')
display(secom_before.groupby('model')['complete'].agg(['sum', 'count']))
print(f"SECOM 已完成 {int(secom_before.complete.sum())}/{len(secom_before)}")

In [ ]:
MAX_JOBS_SECOM = None
secom_oof, secom_folds = run_secom_nested_formal(
    data_dir=SECOM_DIR, frozen_selection_path=SECOM_FROZEN,
    artifact_dir=ARTIFACT_ROOT, seeds=NESTED_FORMAL_SEEDS,
    balance_seed=2026, outer_seed=4096,
    max_jobs=MAX_JOBS_SECOM, device=DEVICE,
)
print(f'SECOM 折级结果 {len(secom_folds)}/100')
print(f'SECOM 种子级 OOF 结果 {len(secom_oof)}/20')
display(secom_oof)

In [ ]:
secom_after = formal_progress(ARTIFACT_ROOT, 'secom')
require_formal_complete(secom_after)
SECOM_RESULT_DIR = ARTIFACT_ROOT / 'secom'
secom_oof = pd.read_csv(SECOM_RESULT_DIR / 'oof_results.csv')
secom_summary = pd.read_csv(SECOM_RESULT_DIR / 'summary.csv')
display(secom_summary)
secom_paired_acc = paired_comparisons(secom_oof, metric='accuracy')
secom_paired_f1 = paired_comparisons(secom_oof, metric='macro_f1')
secom_paired_acc.to_csv(SECOM_RESULT_DIR / 'paired_accuracy.csv', index=False)
secom_paired_f1.to_csv(SECOM_RESULT_DIR / 'paired_macro_f1.csv', index=False)
display(secom_paired_acc)
display(secom_paired_f1)

In [ ]:
ax = secom_summary.set_index('model')[['accuracy_mean', 'macro_f1_mean']].plot.bar(
    figsize=(9, 4), ylim=(0, 1), rot=15, grid=True
)
ax.set_ylabel('score')
ax.set_title('Frozen nested SECOM 5-fold OOF formal evaluation')
plt.tight_layout()
plt.savefig(SECOM_RESULT_DIR / 'formal_accuracy_macro_f1.png', dpi=300, bbox_inches='tight')
plt.show()

## 结果解读规则

必须同时报告全部5个新种子、均值±标准差、配对差值、95%置信区间和胜率。如果 QTrans 只超过部分基线，必须保留最强基线；不得删除 MLP-Mixer、挑选单个种子或根据测试结果再改轮数。

## 输出位置

```text
artifacts/balanced_binary_nested_formal/
├── ucr_wafer/
│   ├── results.csv
│   ├── summary.csv
│   └── paired_macro_f1.csv
└── secom/
    ├── fold_results.csv
    ├── oof_results.csv
    ├── summary.csv
    └── paired_macro_f1.csv
```